# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Purpose of Data Acquisition
This notebook provides a structured and reproducible workflow for acquiring and documenting raw datasets for the research project. 
It helps establish what datasets are acquired, their sources, licenses, formats, schemas, and potential relevancy to Indonesian cyberbullying classification.

## Role in the Full Pipeline
This stage **does not modify** the original datasets. Raw datasets must be preserved exactly as they were acquired.

The pipeline is:
Data Acquisition ➔ Data Merging ➔ EDA ➔ Relabeling ➔ Validation ➔ Preprocessing ➔ TF-IDF ➔ Model Training

# 2. ACQUISITION SCOPE

### Included
- Collecting raw datasets.
- Preserving original files.
- Recording source information.
- Recording licenses.
- Inspecting original schemas.
- Checking whether the dataset is potentially relevant.

### Excluded
- Data cleaning.
- Text normalization.
- Label normalization.
- Duplicate removal.
- Relabeling.
- Final validation.
- EDA.
- Model training.

In [1]:
# 3. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import pathlib
import json
import os
import hashlib
import datetime
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR = BASE_REPORTS_DIR / "data_acquisition"
DATASET_INVENTORY_PATH = REPORTS_DIR / "dataset_inventory.csv"
DATASET_METADATA_PATH = REPORTS_DIR / "dataset_metadata.csv"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Directory: {DATA_RAW_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

Project Root: /home/zapp/Kampus/PM-NEW
Raw Data Directory: /home/zapp/Kampus/PM-NEW/data/raw
Reports Directory: /home/zapp/Kampus/PM-NEW/reports


In [3]:
# 5. RAW DATA DIRECTORY SETUP

# Create directories if they do not exist
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw Directory Path: {DATA_RAW_DIR}")
print(f"Raw Directory Exists: {DATA_RAW_DIR.exists()}")

if DATA_RAW_DIR.exists():
    files_found = [f.name for f in DATA_RAW_DIR.iterdir() if f.is_file() and not f.name.startswith('.')]
    print(f"Number of files found: {len(files_found)}")
    for f in files_found:
        print(f" - {f}")

Raw Directory Path: /home/zapp/Kampus/PM-NEW/data/raw
Raw Directory Exists: True
Number of files found: 6
 - cyberbullying_cleaned_indo.csv
 - kamus_singkatan.csv
 - indotoxic2024_annotated_data_v2_final.csv
 - new_kamusalay.csv
 - data.csv
 - abusive.csv


In [4]:
# 6. RAW DATASET INVENTORY

def calculate_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

inventory_data = []

if DATA_RAW_DIR.exists():
    for idx, filepath in enumerate(sorted(DATA_RAW_DIR.iterdir())):
        if filepath.is_file() and not filepath.name.startswith('.'):
            stat = filepath.stat()
            inventory_data.append({
                "dataset_id": f"DS_{idx+1:02d}",
                "file_name": filepath.name,
                "extension": filepath.suffix,
                "file_size_bytes": stat.st_size,
                "file_path": str(filepath.relative_to(PROJECT_ROOT)),
                "last_modified": datetime.datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
                "file_hash": calculate_sha256(filepath)
            })

df_inventory = pd.DataFrame(inventory_data)
display(df_inventory)

,dataset_id,file_name,extension,file_size_bytes,file_path,last_modified,file_hash
0,DS_02,abusive.csv,.csv,975,data/raw/abusive.csv,2026-07-15 02:52:14,d82db41526e4065fb3ff2825b6ce859339cd4780239646...
1,DS_03,cyberbullying_cleaned_indo.csv,.csv,662886,data/raw/cyberbullying_cleaned_indo.csv,2026-07-09 06:37:10,a43326cbcf54fdc399aeca6390dba94e204414ff252d99...
2,DS_04,data.csv,.csv,1858473,data/raw/data.csv,2020-04-15 14:27:10,44c04e31ad4b7ee4a95f1884e7af4da2c44b69762143eb...
3,DS_05,indotoxic2024_annotated_data_v2_final.csv,.csv,13589262,data/raw/indotoxic2024_annotated_data_v2_final...,2026-07-15 02:57:24,64e6f1b317f2aa05ae6077a4a75821715af5dcc8aac6d3...
4,DS_06,kamus_singkatan.csv,.csv,28690,data/raw/kamus_singkatan.csv,2026-07-09 06:35:53,044adddb761c57ecb3e8e02acf2c0e1f93fc27c740f1b8...
5,DS_07,new_kamusalay.csv,.csv,285941,data/raw/new_kamusalay.csv,2026-07-15 02:52:20,fe00f0c728ee8ecc40dca3c3001ff2701dba7d4417a417...


In [5]:
# 7. DATASET LOADING

dataset_dataframes = {}

for _, row in df_inventory.iterrows():
    filepath = PROJECT_ROOT / row['file_path']
    file_ext = row['extension'].lower()
    dataset_id = row['dataset_id']
    file_name = row['file_name']
    
    print(f"Attempting to load: {file_name}")
    try:
        if file_ext == '.csv':
            try:
                df = pd.read_csv(filepath)
            except UnicodeDecodeError:
                print(f"Retrying {file_name} with latin1 encoding...")
                df = pd.read_csv(filepath, encoding='latin1')
        elif file_ext in ['.xls', '.xlsx']:
            df = pd.read_excel(filepath)
        elif file_ext == '.json':
            df = pd.read_json(filepath)
        elif file_ext == '.parquet':
            df = pd.read_parquet(filepath)
        else:
            print(f"Unsupported format for {file_name}. Skipping.")
            continue
            
        dataset_dataframes[dataset_id] = df
        print(f"Loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
        display(df.head(3))
        print("-" * 50)
    except Exception as e:
        print(f"Error loading {file_name}: {e}")
        print("-" * 50)

Attempting to load: abusive.csv
Loaded successfully: 125 rows, 1 columns
Columns: ['ABUSIVE']


,ABUSIVE
0,alay
1,ampas
2,buta


--------------------------------------------------
Attempting to load: cyberbullying_cleaned_indo.csv
Loaded successfully: 2400 rows, 3 columns
Columns: ['tweet_text_id', 'cyberbullying_type', 'clean_text']


,tweet_text_id,cyberbullying_type,clean_text
0,Setiap orang adalah seorang gadis yang akan me...,age,setiap orang adalah seorang gadis yang akan me...
1,Bahwa pos ab kpop Stans pergi ke sekolah bersa...,age,bahwa pos ab kpop stans pergi ke sekolah bersa...
2,Karena beberapa orang tidak ada yang lebih bai...,age,karena beberapa orang tidak ada yang lebih bai...


--------------------------------------------------
Attempting to load: data.csv
Retrying data.csv with latin1 encoding...
Loaded successfully: 13169 rows, 13 columns
Columns: ['Tweet', 'HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']


...

--------------------------------------------------
Attempting to load: kamus_singkatan.csv
Loaded successfully: 1503 rows, 3 columns
Columns: ['Unnamed: 0', 'singkatan', 'asli']


,Unnamed: 0,singkatan,asli
0,0,abgny,abangnya
1,1,abis,habis
2,2,ad,ada


--------------------------------------------------
Attempting to load: new_kamusalay.csv
Retrying new_kamusalay.csv with latin1 encoding...
Loaded successfully: 15166 rows, 2 columns
Columns: ['anakjakartaasikasik', 'anak jakarta asyik asyik']


,anakjakartaasikasik,anak jakarta asyik asyik
0,pakcikdahtua,pak cik sudah tua
1,pakcikmudalagi,pak cik muda lagi
2,t3tapjokowi,tetap jokowi


--------------------------------------------------


In [6]:
# 8. ORIGINAL DATASET STRUCTURE INSPECTION

# This is a basic acquisition-level inspection, not full EDA.
for ds_id, df in dataset_dataframes.items():
    print(f"Dataset ID: {ds_id}")
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")
    
    # Inspect Data Types and Missing Values
    summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
    })
    display(summary)
    
    # Unique values for categorical/object columns (max 5 values shown for brevity)
    print("Categorical Unique Values Sample:")
    for col in df.select_dtypes(include=['object', 'category', 'string']).columns:
        unique_vals = df[col].dropna().unique()
        sample_vals = [str(x)[:100] + ("..." if len(str(x)) > 100 else "") for x in unique_vals[:5]]
        print(f" - {col} ({len(unique_vals)} unique): {sample_vals}")
    
    print("=" * 60)

Dataset ID: DS_02
Number of rows: 125
Number of columns: 1


,Data Type,Missing Values,Missing %
ABUSIVE,str,0,0.0


Categorical Unique Values Sample:
 - ABUSIVE (125 unique): ['alay', 'ampas', 'buta', 'keparat', 'anjing']
Dataset ID: DS_03
Number of rows: 2400
Number of columns: 3


,Data Type,Missing Values,Missing %
tweet_text_id,str,0,0.00
cyberbullying_type,str,0,0.00
clean_text,str,9,0.38


Categorical Unique Values Sample:
 - tweet_text_id (2397 unique): ['Setiap orang adalah seorang gadis yang akan mengganggu saya di sekolah tinggi.', 'Bahwa pos ab kpop Stans pergi ke sekolah bersama-sama dan semua orang mengatakan mereka akan mengger...', 'Karena beberapa orang tidak ada yang lebih baik untuk dilakukan, atau mereka adalah pengganggu di se...', 'Bro aku pelatih JV tahun lalu di Skyline.. dan aku harus mengubah air menjadi anggur kami memenangka...', 'Wanita-wanita ini benar-benar mengingatkan saya pada anak ayam SMA ... dengan semua penindasan oleh ...']
 - cyberbullying_type (6 unique): ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying']
 - clean_text (2382 unique): ['setiap orang adalah seorang gadis yang akan mengganggu saya di sekolah tinggi', 'bahwa pos ab kpop stans pergi ke sekolah bersamasama dan semua orang mengatakan mereka akan menggert...', 'karena beberapa orang tidak ada yang lebih baik untuk dilakukan atau mereka adalah pengganggu d

...

Categorical Unique Values Sample:
 - text_id (28448 unique): ['1-1', '1-10', '1-100', '1-1000', '1-1001']
 - annotators_id (102 unique): ["['7', '15']", "['10', '18']", "['9', '8']", "['2', '12']", "['2', '12', '30', '24']"]
 - text (26173 unique): ['Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023', 'Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, po...', 'Gini aja deh @KPU_ID usul aja nih. Gimana kalo misalnya gak usah ada debat aja kalo ga boleh dikriti...', 'Relawan Tionghoa Kalbar yg di Jakarta mendeklarasikan dukungan kepada Ganjar Pranowo. Mario Dandy Bu...', 'Kristen, Hindu, Islam dapat perlakuan istimewa dari pak Anies Ncep ketar-ketir']
 - initial_paragraph (1599 unique): ['“️Jurnalis Hana Mahamed kembali tampil di televisi dengan wajah terluka akibat serangan Israel di Ga...', '"Wanita Islam melecehkan seorang wanita Hindu karena berani naik bus di Kerala tanpa penutup syariah...', 'NARASI: “JAGA² SAJA,JIKA 

,Data Type,Missing Values,Missing %
Unnamed: 0,int64,0,0.0
singkatan,str,0,0.0
asli,str,0,0.0


Categorical Unique Values Sample:
 - singkatan (1496 unique): ['abgny', 'abis', 'ad', 'adek', 'adik2']
 - asli (790 unique): ['abangnya', 'habis', 'ada', 'adik', 'adik-adik']
Dataset ID: DS_07
Number of rows: 15166
Number of columns: 2


,Data Type,Missing Values,Missing %
anakjakartaasikasik,str,0,0.0
anak jakarta asyik asyik,str,0,0.0


Categorical Unique Values Sample:
 - anakjakartaasikasik (15166 unique): ['pakcikdahtua', 'pakcikmudalagi', 't3tapjokowi', '3x', 'aamiin']
 - anak jakarta asyik asyik (8638 unique): ['pak cik sudah tua', 'pak cik muda lagi', 'tetap jokowi', 'tiga kali', 'amin']


In [7]:
# 9. DATASET COLUMN IDENTIFICATION

# Potential common names
TEXT_COL_CANDIDATES = ['text', 'comment', 'content', 'tweet', 'message', 'sentence', 'review']
LABEL_COL_CANDIDATES = ['label', 'class', 'category', 'type', 'cyberbullying_type', 'severity', 'severity_level']

column_candidates = {}

for ds_id, df in dataset_dataframes.items():
    cols = [str(c).lower() for c in df.columns]
    
    text_cands = [c for c in cols if any(cand in c for cand in TEXT_COL_CANDIDATES)]
    label_cands = [c for c in cols if any(cand in c for cand in LABEL_COL_CANDIDATES)]
    
    column_candidates[ds_id] = {
        "text_candidates": text_cands,
        "label_candidates": label_cands
    }
    
    print(f"Dataset {ds_id} Candidates:")
    print(f" - Potential Text Columns: {text_cands}")
    print(f" - Potential Label Columns: {label_cands}")
    print("-" * 40)

Dataset DS_02 Candidates:
 - Potential Text Columns: []
 - Potential Label Columns: []
----------------------------------------
Dataset DS_03 Candidates:
 - Potential Text Columns: ['tweet_text_id', 'clean_text']
 - Potential Label Columns: ['cyberbullying_type']
----------------------------------------
Dataset DS_04 Candidates:
 - Potential Text Columns: ['tweet']
 - Potential Label Columns: []
----------------------------------------
Dataset DS_05 Candidates:
 - Potential Text Columns: ['text_id', 'text', 'is_noise_or_spam_text']
 - Potential Label Columns: []
----------------------------------------
Dataset DS_06 Candidates:
 - Potential Text Columns: []
 - Potential Label Columns: []
----------------------------------------
Dataset DS_07 Candidates:
 - Potential Text Columns: []
 - Potential Label Columns: []
----------------------------------------


In [8]:
# 10. DATASET METADATA REGISTRATION

# This registry should be manually updated by the user for each acquired dataset.
# The following is a template populated with placeholders.
dataset_metadata = []

for ds_id in dataset_dataframes.keys():
    df = dataset_dataframes[ds_id]
    cands = column_candidates.get(ds_id, {})
    
    metadata_entry = {
        "dataset_id": ds_id,
        "file_name": df_inventory.loc[df_inventory['dataset_id'] == ds_id, 'file_name'].values[0] if not df_inventory.empty else "Unknown",
        "source_name": "To be documented", 
        "source_url": "To be documented",
        "dataset_description": "To be documented",
        "original_language": "Indonesian", # Verify manually
        "original_format": df_inventory.loc[df_inventory['dataset_id'] == ds_id, 'extension'].values[0] if not df_inventory.empty else "Unknown",
        "license": "To be documented",
        "license_url": "To be documented",
        "acquisition_date": datetime.datetime.now().strftime('%Y-%m-%d'),
        "original_row_count": df.shape[0],
        "original_column_count": df.shape[1],
        "text_column_candidate": cands.get("text_candidates", ["Unknown"])[0] if cands.get("text_candidates") else "Unknown",
        "label_column_candidate": cands.get("label_candidates", ["Unknown"])[0] if cands.get("label_candidates") else "Unknown",
        "type_label_available": "Needs Review", # Set True/False manually
        "severity_label_available": "Needs Review", # Set True/False manually
        "notes": "None",
        "limitations": "None"
    }
    dataset_metadata.append(metadata_entry)

df_metadata = pd.DataFrame(dataset_metadata)
display(df_metadata)

,dataset_id,file_name,source_name,source_url,dataset_description,original_language,original_format,license,license_url,acquisition_date,original_row_count,original_column_count,text_column_candidate,label_column_candidate,type_label_available,severity_label_available,notes,limitations
0,DS_02,abusive.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,125,1,Unknown,Unknown,Needs Review,Needs Review,None,None
1,DS_03,cyberbullying_cleaned_indo.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,2400,3,tweet_text_id,cyberbullying_type,Needs Review,Needs Review,None,None
2,DS_04,data.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,13169,13,tweet,Unknown,Needs Review,Needs Review,None,None
3,DS_05,indotoxic2024_annotated_data_v2_final.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,28448,14,text_id,Unknown,Needs Review,Needs Review,None,None
4,DS_06,kamus_singkatan.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,1503,3,Unknown,Unknown,Needs Review,Needs Review,None,None
5,DS_07,new_kamusalay.csv,To be documented,To be documented,To be documented,Indonesian,.csv,To be documented,To be documented,2026-07-19,15166,2,Unknown,Unknown,Needs Review,Needs Review,None,None


# 11. DATASET SOURCE AND LICENSE DOCUMENTATION

The dataset source and license must be documented accurately to ensure research reproducibility and attribution.
Please manually fill in the `source_name`, `source_url`, `license`, and `license_url` in the metadata registry above for each dataset you acquire.

**Do not invent source information.** If a dataset is from Kaggle, link to the Kaggle dataset. If it's from a GitHub repository, link to the repository and explicitly state its license (e.g., MIT, CC-BY 4.0).

In [9]:
# 12. DATASET SUITABILITY SCREENING

suitability_data = []

for idx, row in df_metadata.iterrows():
    ds_id = row['dataset_id']
    
    # Preliminary checks based on metadata and candidates
    has_indonesian = row['original_language'].lower() == 'indonesian'
    has_text_cand = row['text_column_candidate'] != 'Unknown'
    has_label_cand = row['label_column_candidate'] != 'Unknown'
    
    # Potentially relevant if it has text, label, and is indonesian
    potentially_relevant = "Yes" if (has_indonesian and has_text_cand and has_label_cand) else "Needs Review"
    
    suitability_data.append({
        "Dataset": ds_id,
        "Indonesian Text": "Yes" if has_indonesian else "Unknown",
        "Text Candidate": "Yes" if has_text_cand else "No",
        "Label Candidate": "Yes" if has_label_cand else "No",
        "Type Available": row['type_label_available'],
        "Severity Available": row['severity_label_available'],
        "Potentially Relevant": potentially_relevant
    })

df_suitability = pd.DataFrame(suitability_data)
display(df_suitability)

,Dataset,Indonesian Text,Text Candidate,Label Candidate,Type Available,Severity Available,Potentially Relevant
0,DS_02,Yes,No,No,Needs Review,Needs Review,Needs Review
1,DS_03,Yes,Yes,Yes,Needs Review,Needs Review,Yes
2,DS_04,Yes,Yes,No,Needs Review,Needs Review,Needs Review
3,DS_05,Yes,Yes,No,Needs Review,Needs Review,Needs Review
4,DS_06,Yes,No,No,Needs Review,Needs Review,Needs Review
5,DS_07,Yes,No,No,Needs Review,Needs Review,Needs Review


In [10]:
# 13. ACQUISITION QUALITY CHECKS

quality_checks = []

for _, row in df_inventory.iterrows():
    ds_id = row['dataset_id']
    file_path = PROJECT_ROOT / row['file_path']
    
    exists = file_path.exists()
    readable = os.access(file_path, os.R_OK) if exists else False
    non_empty = file_path.stat().st_size > 0 if exists else False
    
    has_rows = False
    has_cols = False
    
    if ds_id in dataset_dataframes:
        df = dataset_dataframes[ds_id]
        has_rows = df.shape[0] > 0
        has_cols = df.shape[1] > 0
        
    status = "Pass" if (exists and readable and non_empty and has_rows and has_cols) else "Needs Review"
    
    quality_checks.append({
        "Dataset": ds_id,
        "File Exists": exists,
        "Readable": readable,
        "Non-Empty": non_empty,
        "Has Columns": has_cols,
        "Status": status
    })

df_quality = pd.DataFrame(quality_checks)
display(df_quality)

,Dataset,File Exists,Readable,Non-Empty,Has Columns,Status
0,DS_02,True,True,True,True,Pass
1,DS_03,True,True,True,True,Pass
2,DS_04,True,True,True,True,Pass
3,DS_05,True,True,True,True,Pass
4,DS_06,True,True,True,True,Pass
5,DS_07,True,True,True,True,Pass


In [11]:
# 14. DATASET INVENTORY EXPORT

if not df_inventory.empty:
    df_inventory.to_csv(DATASET_INVENTORY_PATH, index=False)
    print(f"Exported Dataset Inventory to: {DATASET_INVENTORY_PATH}")

if not df_metadata.empty:
    df_metadata.to_csv(DATASET_METADATA_PATH, index=False)
    print(f"Exported Dataset Metadata to: {DATASET_METADATA_PATH}")

# Also write an acquisition summary markdown file if needed
summary_md_path = REPORTS_DIR / "acquisition_summary.md"
with open(summary_md_path, "w") as f:
    f.write("# Acquisition Summary\n\n")
    f.write(f"- Total Datasets Acquired: {len(df_inventory)}\n")
    f.write("- Dataset Formats: " + ", ".join(df_inventory['extension'].unique() if not df_inventory.empty else []) + "\n")
    
print(f"Exported Acquisition Summary to: {summary_md_path}")

Exported Dataset Inventory to: /home/zapp/Kampus/PM-NEW/reports/dataset_inventory.csv
Exported Dataset Metadata to: /home/zapp/Kampus/PM-NEW/reports/dataset_metadata.csv
Exported Acquisition Summary to: /home/zapp/Kampus/PM-NEW/reports/acquisition_summary.md


# 15. ACQUISITION SUMMARY

### Total Datasets Acquired
Review the exported `dataset_inventory.csv` for the complete list of files successfully detected in `data/raw/`.

### Dataset Formats
Supported formats currently include `.csv`, `.xlsx`, `.json`, and `.parquet`. Unsupported formats are logged and skipped.

### Potentially Relevant Datasets
Check the **Dataset Suitability Screening** table to see which datasets have both text and label candidates in Indonesian.

### Dataset Limitations
Ensure you have manually filled in any known limitations, such as missing labels, missing severity information, or unknown licenses in the **Dataset Metadata Registration** cell before proceeding.

### Next Step
The next notebook is:
`notebooks/01_data_merging.ipynb`

The next stage will:
- Select relevant datasets.
- Map compatible columns.
- Merge datasets.
- Preserve source provenance.
- Produce the merged dataset.

# 16. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Place manually downloaded raw datasets into: `data/raw/`
3. Open: `notebooks/00_data_acquisition.ipynb`
4. Review and complete the **Dataset Metadata Registration** section manually (add licenses, source URLs, etc.).
5. Run the notebook from the first cell to the last cell.
6. Review the dataset inventory and acquisition summary exported to the `reports/` folder.
7. Confirm that relevant datasets are ready for: `notebooks/01_data_merging.ipynb`